In [2]:
import nest_asyncio
nest_asyncio.apply()  # 중첩 이벤트 루프 허용
import os
import json
import logging
import asyncio
import pandas as pd
import numpy as np
import openai
from datetime import datetime
import sys
from typing import List, Dict, Optional
# from tenacity import retry, stop_after_attempt, wait_exponential
import re
import gc
import matplotlib.pyplot as plt
from tqdm import tqdm
api = pd.read_csv('../../data/info.csv')
api_key = api.loc[1][1]


/var/folders/7z/944bgjcs659fsp58nc954qgm0000gn/T/ipykernel_7906/755927677.py:19: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  api_key = api.loc[1][1]


In [3]:
sens = pd.read_parquet('/Users/nam-yeong/git/prj_centum/gpt_word/final_result/final_result_20250410_060315.parquet')
nums = pd.read_parquet('../../data/centum_data_numeric_cleaned.parquet')

sens['날짜'] = pd.to_datetime(sens['날짜'])
df = pd.merge(nums, sens, on=['환자번호','날짜'], how='left')

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
df = pd.merge(nums, sens, on=['환자번호','날짜'], how='left')
cols = [
       '환자번호', '날짜', 'CC', '약', '장치', '습관', '찜질', '마사지, 스트레칭', 'PI',
       'CMO', 'MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading',
       'Occlusion', 'OJ/OB', 'Class', 'Midline Shift', 'Deviation', 'CR-CO',
       'Tongue ridging', 'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt', '치료계획',
       'End feel',
       'CC_location', 'CC_pain_type', 'CC_painUncomp_desc_jaw',
       'CC_disable_desc_jaw', 'CC_muscle_joint_desc_stress',
       'CC_dentalHistory_desc', 'CC_clinic_history_desc', 'CC_factor_habbit',
       'CC_treat_plan', 'CC_severity', 'CC_duration',
       '약_medication_type', '약_frequency', '약_duration', '약_compliance',
       '장치_device_type', '장치_usage_pattern', '장치_duration', '장치_compliance',
       '습관_habit_type', '습관_frequency', '습관_awareness', '습관_improvement',
       '찜질_status', '찜질_frequency', '찜질_duration', '찜질_method',
       '마사지, 스트레칭_type', '마사지, 스트레칭_frequency', '마사지, 스트레칭_duration',
       '마사지, 스트레칭_method',
       'CMO_before', 'CMO_after', 'MMO_before', 'MMO_after',
       'deviation_pattern_type', 'deviation_direction', 'deviation_intensity',
       'Cap.pal_Pain_Intensity', 'Cap.pal_Pain_Direction',
       'Cap.pal_Pain_Situation', 'M.pal_Pain_Intensity',
       'M.pal_Pain_Direction', 'M.pal_Pain_Situation', 'Noise_Code',
       'Noise_Direction', 'Noise_Intensity', 'Noise_Situation',
       'Occlusion_lt_number', 'Occlusion_rt_number', 'Occlusion_lt_Intensity',
       'Occlusion_rt_Intensity', 'oj', 'ob', 'Midline_Shift_Jaw',
       'Midline_Shift_Direction_x', 'Midline_Shift_Direction_y',
       'Midline_Shift_Amount', 'CRCO_Direction_x', 'CRCO_Direction_y',
       'CRCO_Amount', 'Tongue_ridging_Intensity', 'Mucosal_ridging_Intensity', 'Rt_before', 'Rt_after',
       'Lt_before', 'Lt_after', 'Next_Visit_Days'
]
df = df[cols]

# df.to_parquet(f'../../data/final_without_pi_centum_data_with_medical_data_{timestamp}.parquet')

# df = pd.read_parquet('../../data/final_without_pi_centum_data_with_medical_data_20250411_225729.parquet')
del sens, nums
gc.collect()

0

In [4]:
## NaN 처리 

df = df.replace('', np.nan)
df = df.replace('-', np.nan)
df = df.replace('- -', np.nan)
df = df.replace('n/s', np.nan)

df['약_medication_type'] = df['약_medication_type'].replace('없음',np.nan)
df['약_medication_type'] = df['약_medication_type'].replace('약','약물 종류 미상')
df['약_medication_type'] = df['약_medication_type'].replace('약물 종류','약물 종류 미상')
df['약_medication_type'] = df['약_medication_type'].replace('약물 종류','약물 종류 미상')
df['약_medication_type'] = df['약_medication_type'].replace('다복용함','약물 종류 미상')
df['약_medication_type'] = df['약_medication_type'].replace('다복용','약물 종류 미상')
df['약_medication_type'] = df['약_medication_type'].replace('저녁약','약물 종류 미상')
df['약_medication_type'] = df['약_medication_type'].replace('다양한 약물','약물 종류 미상')

df['Noise_Code'] = df['Noise_Code'].apply(lambda x : "No-Noise" if x == 0 else "Click" if x == 1 else "Popping" if x == 2 else "Crepitus" if x == 3 else "Unknown")

text_cols = [
    'CC_location','CC_pain_type','CC_painUncomp_desc_jaw','CC_disable_desc_jaw','CC_muscle_joint_desc_stress',
    'CC_dentalHistory_desc','CC_clinic_history_desc','CC_factor_habbit','CC_treat_plan',
    '약_medication_type','약_compliance','장치_device_type','습관_habit_type','습관_awareness'
    ]
numeric_cols = [
    'CC_duration','CC_severity', 'CC_vas', 'CMO_before','CMO_after','MMO_before','MMO_after','Midline_Shift_Amount','CRCO_Amount','Next_Visit_Days'
    ,'Rt_before','Rt_after','Lt_before','Lt_after','Tongue_ridging_Intensity','Mucosal_ridging_Intensity' ,'장치_duration','찜질_duration','마사지, 스트레칭_duration','약_duration'
    ,'M.pal_Pain_Intensity','Cap.pal_Pain_Intensity','Noise_Intensity','Occlusion_lt_Intensity','Occlusion_rt_Intensity', 'oj','ob'
    ]
category_cols = [
    '장치_usage_pattern','장치_compliance', '습관_frequency', '습관_improvement','약_frequency',
    '찜질_status','찜질_frequency', '마사지, 스트레칭_frequency','마사지, 스트레칭_method' ,'deviation_pattern_type','deviation_direction',
    'Cap.pal_Pain_Direction','Cap.pal_Pain_Situation','M.pal_Pain_Direction','M.pal_Pain_Situation','Noise_Code','Noise_Direction',
    'Noise_Situation','Occlusion_lt_number','Occlusion_rt_number','Midline_Shift_Jaw','Midline_Shift_Direction_x','Midline_Shift_Direction_y',
    'CRCO_Direction_x','CRCO_Direction_y','Cap.pal_Pain_Intensity','deviation_intensity',
    '마사지, 스트레칭_frequency','마사지, 스트레칭_type','찜질_method'
    ]


In [5]:
def extract_vas_values(df, source_col='CC', target_col='CC_vas'):
    """
    통합된 VAS 추출 및 처리 함수 - 수동 보정 포함
    """
    # 결과 컬럼 초기화
    df[target_col] = np.nan
    
    # 1. 정규식 패턴 정의 - 다양한 VAS 표현 형식 포괄
    vas_patterns = [
        r'(?i)vas\D*(\d+(?:\.\d+)?)',  # 기본 VAS 패턴
        r'(?i)통증점수\D*(\d+(?:\.\d+)?)',  # '통증점수' 포함 패턴
        r'(?i)통증 점수\D*(\d+(?:\.\d+)?)',  # '통증 점수' 포함 패턴
        r'(?i)통증강도\D*(\d+(?:\.\d+)?)',   # '통증강도' 포함 패턴
        r'(?i)점수[는은이가]\D*(\d+(?:\.\d+)?)'  # '점수는/은/이/가' 포함 패턴
    ]
    
    # 2. 'vas' 포함 행 필터링 (성능 최적화)
    mask = df[source_col].str.contains('|'.join(['vas', 'VAS', '통증점수', '통증 점수', '통증강도']), 
                                      na=False, regex=True)
    filtered = df.loc[mask, source_col]
    
    # 3. 모든 패턴에 대해 VAS 값 추출 시도
    extracted_values = pd.Series([None] * len(filtered), index=filtered.index)
    
    for pattern in vas_patterns:
        # 아직 값이 추출되지 않은 행에 대해서만 처리
        null_mask = extracted_values.isna()
        if not null_mask.any():
            break
            
        # 패턴 적용 시도
        try:
            pattern_result = filtered[null_mask].str.extract(pattern, expand=False)
            extracted_values.loc[null_mask] = pattern_result.combine_first(extracted_values.loc[null_mask])
        except Exception as e:
            print(f"패턴 '{pattern}' 처리 중 오류 발생: {e}")
    
    # 4. 결과 값 정제 - 숫자로 변환 및 유효 범위 확인
    extracted_values = pd.to_numeric(extracted_values, errors='coerce')
    
    # VAS 범위는 일반적으로 0-10이므로, 범위를 벗어난 값 처리
    extracted_values = extracted_values.apply(
        lambda x: float(x) if pd.notna(x) and 0 <= float(x) <= 100 else np.nan
    )
    
    # 100 초과 값은 잘못 추출된 것으로 간주하고 범위 조정
    extracted_values = extracted_values.apply(
        lambda x: float(x)/10 if pd.notna(x) and 10 < float(x) <= 100 else x
    )
    
    # 5. 추출된 값 원본 데이터프레임에 적용
    df.loc[mask, target_col] = extracted_values
    
    # 6. 제공된 수동 보정 데이터 적용
    # 수동 보정 데이터
    manual_correction_data = {
        'index': [1409, 1815, 3576, 4372, 5344, 9603, 10634, 10767, 12041, 13048,
                  13081, 13218, 14292, 14615, 15034, 15576, 16569, 16730, 16925, 16967,
                  18848, 20403, 21441, 22723, 22804, 23610, 23669, 23815, 24390, 25705,
                  26581, 27379],
        'CC_vas': [3, 3, 7, 1, 1, 3, 8, 7, 2, 6,
                   2, 0, 5, 3, 2, 3, 1, 5, 3, 1,
                   3, 2, 10, 4, 4, 4, 2, 7, 5, 8,
                   0, 3]
    }
    
    # 수동 보정 데이터를 딕셔너리로 변환
    manual_corrections = dict(zip(manual_correction_data['index'], manual_correction_data['CC_vas']))
    
    # 수동 보정 값 적용
    for idx, val in manual_corrections.items():
        if idx in df.index:
            df.loc[idx, target_col] = val
            
    # 7. 데이터 유효성 검사 및 통계
    vas_stats = {
        '자동 추출된 VAS 값 수': mask.sum(),
        '수동 보정된 VAS 값 수': sum(idx in df.index for idx in manual_corrections.keys()),
        '최종 유효 VAS 값 수': df[target_col].notna().sum(),
        'VAS 값 범위': f"{df[target_col].min()} ~ {df[target_col].max()}" if df[target_col].notna().any() else "N/A"
    }
    print(f"VAS 값 추출 통계: {vas_stats}")
    
    return df

In [6]:
def handle_missing_values(df):
    """
    데이터 유형을 고려한 개선된 결측치 처리 함수
    """
    # 1. 범주형과 수치형 변수 구분 - 존재하는 열만 처리
    numeric_cols = [col for col in df.columns if col in [
        'CC_severity', 'CMO_before', 'MMO_before', 'dif_MMO_CMO'
    ] or col.endswith('_Intensity') or col.endswith('_duration')]
    
    # CC_vas가 있으면 포함
    if 'CC_vas' in df.columns:
        numeric_cols.append('CC_vas')
    
    categorical_cols = [col for col in df.columns if col not in numeric_cols 
                       and not pd.api.types.is_datetime64_any_dtype(df[col])
                       and col != '환자번호']
    
    # 2. 표준화된 결측값 변환
    for replace_val in ['', '-', '- -', 'n/s', '없음']:
        df = df.replace(replace_val, np.nan)
    
    # 3. 수치형 변수 형변환 및 결측치 처리
    for col in numeric_cols:
        # 먼저 숫자형으로 안전하게 변환
        try:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        except Exception as e:
            print(f"컬럼 {col} 변환 중 오류 발생: {e}")
        
        # 동일 환자의 다른 방문 데이터가 있으면 그 값으로 대체
        df[col] = df.groupby('환자번호')[col].transform(
            lambda x: x.fillna(method='ffill').fillna(method='bfill')
        )
        
        # 여전히 결측치가 있는 경우 중앙값으로 대체
        try:
            median_val = df[col].median()
            if pd.notna(median_val):  # None이 아닌지 확인
                df[col] = df[col].fillna(median_val)
        except TypeError:
            # 중앙값을 계산할 수 없는 경우 0으로 대체
            print(f"컬럼 {col}의 중앙값을 계산할 수 없습니다. 0으로 대체합니다.")
            df[col] = df[col].fillna(0)
    
    # 4. 범주형 변수 결측치 처리
    for col in categorical_cols:
        # 중요도가 높은 열만 최빈값으로 대체, 나머지는 'Unknown'으로 대체
        if col in ['CC_location', 'CC_pain_type', 'Noise_Code', '습관_habit_type']:
            try:
                mode_val = df[col].mode().iloc[0] if not df[col].mode().empty else 'Unknown'
                df[col] = df[col].fillna(mode_val)
            except:
                df[col] = df[col].fillna('Unknown')
        else:
            df[col] = df[col].fillna('Unknown')
            
    return df

In [7]:
# 일일 성장률 계산 함수 추가
def add_daily_growth_rate(df, column_name):
    """
    원본 데이터프레임에 일일 성장률 피처를 추가하는 함수
    
    Parameters:
    df (DataFrame): 환자 데이터가 포함된 데이터프레임
    column_name (str): 성장률을 계산할 컬럼명
    
    Returns:
    DataFrame: 일일 성장률 피처가 추가된 데이터프레임
    """
    # 날짜 형식 변환 (이미 변환되어 있지 않은 경우)
    if not pd.api.types.is_datetime64_any_dtype(df['날짜']):
        df['날짜'] = pd.to_datetime(df['날짜'])
    
    # 환자별로 그룹화하여 처리
    df_copy = df.copy()
    
    # 환자별, 날짜별로 정렬
    df_copy = df_copy.sort_values(['환자번호', '날짜'])
    
    # 환자별 첫 방문 값 추출
    first_values = df_copy.groupby('환자번호')[column_name].first().reset_index()
    first_values.columns = ['환자번호', f'첫방문_{column_name}']
    
    # 첫 방문 값을 원본 데이터에 병합
    df_copy = pd.merge(df_copy, first_values, on='환자번호', how='left')
    
    # 일일 성장률 계산 (첫 방문 대비)
    df_copy[f'{column_name}_일일성장률'] = 0.0
    
    # 문자열 데이터 타입 확인 및 숫자로 변환
    numeric_mask = pd.to_numeric(df_copy[column_name], errors='coerce').notna() & pd.to_numeric(df_copy[f'첫방문_{column_name}'], errors='coerce').notna()
    
    # 숫자로 변환 가능한 데이터만 처리
    if numeric_mask.any():
        # 숫자형으로 변환
        current_values = pd.to_numeric(df_copy.loc[numeric_mask, column_name])
        first_visit_values = pd.to_numeric(df_copy.loc[numeric_mask, f'첫방문_{column_name}'])
        
        # 0으로 나누기 방지
        valid_mask = (first_visit_values != 0)
        
        # 유효한 데이터에 대해서만 성장률 계산
        if valid_mask.any():
            df_copy.loc[numeric_mask[numeric_mask].index[valid_mask], f'{column_name}_일일성장률'] = (
                (current_values[valid_mask] - first_visit_values[valid_mask]) / 
                first_visit_values[valid_mask] * 100
            )
    
    return df_copy


In [8]:
def preprocessing_pipeline(df):
    """
    데이터 전처리 통합 파이프라인 - 데이터 타입 처리 강화
    """
    print("1. 데이터 로딩 완료, 행 수:", len(df))
    
    # 1. 기본 결측치 변환 (문자열 -> NaN)
    for replace_val in ['', '-', '- -', 'n/s', '없음']:
        df = df.replace(replace_val, np.nan)
    print("2. 기본 결측치 변환 완료")
    
    # 2. 수치형 열 미리 변환
    numeric_candidates = ['CMO_before', 'MMO_before', 'CC_severity', 'CC_vas']
    for col in numeric_candidates:
        if col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                print(f"컬럼 {col} 숫자형 변환 완료")
            except Exception as e:
                print(f"컬럼 {col} 변환 실패: {e}")
    
    # 3. VAS 값 추출 및 정제
    df = extract_vas_values(df)
    print("3. VAS 값 추출 완료")
    
    # 4. 고급 결측치 처리
    df = handle_missing_values(df)
    print("4. 고급 결측치 처리 완료")
    
    # 5. MMO-CMO 차이 계산
    if 'MMO_before' in df.columns and 'CMO_before' in df.columns:
        # 계산 전 다시 한번 숫자형 확인
        df['MMO_before'] = pd.to_numeric(df['MMO_before'], errors='coerce')
        df['CMO_before'] = pd.to_numeric(df['CMO_before'], errors='coerce')
        df['dif_MMO_CMO'] = df['MMO_before'] - df['CMO_before']
        print("5. MMO-CMO 차이 계산 완료")
    
    # 6. 성장률 계산
    growth_columns = ['CMO_before', 'CC_vas', 'MMO_before', 'dif_MMO_CMO']
    for column in growth_columns:
        if column in df.columns:
            try:
                df = add_daily_growth_rate(df, column)
                print(f"컬럼 {column} 성장률 계산 완료")
            except Exception as e:
                print(f"컬럼 {column} 성장률 계산 실패: {e}")
    
    # 7. 오류 데이터 필터링
    if 'dif_MMO_CMO' in df.columns:
        # 깊은 복사로 경고 방지
        orig_len = len(df)
        df = df.query('dif_MMO_CMO >= 0').copy()
        print(f"7. 오류 데이터 필터링: {orig_len - len(df)}개 행 제거됨, 남은 행 수: {len(df)}")
    
    return df

In [9]:
df = preprocessing_pipeline(df)

1. 데이터 로딩 완료, 행 수: 28162
2. 기본 결측치 변환 완료
컬럼 CMO_before 숫자형 변환 완료
컬럼 MMO_before 숫자형 변환 완료
컬럼 CC_severity 숫자형 변환 완료
VAS 값 추출 통계: {'자동 추출된 VAS 값 수': 14108, '수동 보정된 VAS 값 수': 32, '최종 유효 VAS 값 수': 14101, 'VAS 값 범위': '0.0 ~ 10.0'}
3. VAS 값 추출 완료


/var/folders/7z/944bgjcs659fsp58nc954qgm0000gn/T/ipykernel_7906/147469774.py:32: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  lambda x: x.fillna(method='ffill').fillna(method='bfill')
/var/folders/7z/944bgjcs659fsp58nc954qgm0000gn/T/ipykernel_7906/147469774.py:32: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  lambda x: x.fillna(method='ffill').fillna(method='bfill')
/var/folders/7z/944bgjcs659fsp58nc954qgm0000gn/T/ipykernel_7906/147469774.py:32: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  lambda x: x.fillna(method='ffill').fillna(method='bfill')
/Users/nam-yeong/miniforge3/envs/pymc_env/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, k

4. 고급 결측치 처리 완료
5. MMO-CMO 차이 계산 완료
컬럼 CMO_before 성장률 계산 완료
컬럼 CC_vas 성장률 계산 완료
컬럼 MMO_before 성장률 계산 완료
컬럼 dif_MMO_CMO 성장률 계산 완료
7. 오류 데이터 필터링: 262개 행 제거됨, 남은 행 수: 27900


ArrowTypeError: ("Expected bytes, got a 'float' object", 'Conversion failed for column 찜질_status with type object')

In [19]:
import h5py
df.to_hdf('/Users/nam-yeong/git/prj_centum/gpt_word/final_result/preprocessed_final_df.h5', 
          key='df', 
          mode='w',
          complevel=9,  # 최대 압축
          complib='blosc')  # 빠른 압축 라이브러리

/var/folders/7z/944bgjcs659fsp58nc954qgm0000gn/T/ipykernel_7906/3800509428.py:3: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block3_values] [items->Index(['환자번호', 'CC', '약', '장치', '습관', '찜질', '마사지, 스트레칭', 'PI', 'CMO', 'MMO',
       'Cap.pal', 'M.pal', 'Noise', 'Loading', 'Occlusion', 'OJ/OB', 'Class',
       'Midline Shift', 'Deviation', 'CR-CO', 'Tongue ridging',
       'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt', '치료계획', 'End feel',
       'CC_location', 'CC_pain_type', 'CC_painUncomp_desc_jaw',
       'CC_disable_desc_jaw', 'CC_muscle_joint_desc_stress',
       'CC_dentalHistory_desc', 'CC_clinic_history_desc', 'CC_factor_habbit',
       'CC_treat_plan', '약_medication_type', '약_frequency', '약_compliance',
       '장치_device_type', '장치_usage_pattern', '장치_compliance', '습관_habit_type',
       '습관_frequency', '습관_awareness', '습관_improvement', '찜질_status',
       '찜질_frequency', '찜질

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 27900 entries, 0 to 28161
Columns: 106 entries, 환자번호 to dif_MMO_CMO_일일성장률
dtypes: datetime64[ns](1), float64(18), int64(9), object(78)
memory usage: 22.8+ MB
